Name:- Tanmay Santosh Sanap

Roll No:-40

Student Id:- 5144978

College:- B K BIRLA COLLEGE

Subject:-Data Engineering (Practical No :- 08)

To implement PySpark DataFrame operations for reading and analyzing CSV data, including schema inspection, filtering, grouping, aggregation, duplicate removal, joining datasets, and calculating average sales by product category.

In [1]:
# ============================================================
# PRACTICAL 8 - PYSPARK DATA OPERATIONS
# ============================================================

# If PySpark is not installed in Google Colab, run this first:
# !pip install pyspark -q


from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, round
import os


print("=" * 60)
print("             PRACTICAL 8 - PYSPARK")
print("=" * 60)


# ============================================================
# 0. SETUP: INITIALIZE SPARK
# ============================================================

spark = SparkSession.builder \
    .appName("PySpark_Data_Operations") \
    .master("local[*]") \
    .getOrCreate()


# ============================================================
# CREATE SALES CSV FILE
# ============================================================

with open("sales.csv", "w") as f:

    f.write("transaction_id,product_id,quantity,sale_amount\n")
    f.write("1,101,2,50.0\n")
    f.write("2,102,1,120.0\n")
    f.write("3,101,4,100.0\n")
    f.write("3,101,4,100.0\n")   # Duplicate
    f.write("4,103,1,30.0\n")
    f.write("5,101,0,0.0\n")     # Invalid quantity


# ============================================================
# CREATE PRODUCTS CSV FILE
# ============================================================

with open("products.csv", "w") as f:

    f.write("product_id,product_name,category\n")
    f.write("101,Wireless Mouse,Electronics\n")
    f.write("102,Mechanical Keyboard,Electronics\n")
    f.write("103,Coffee Mug,Home & Kitchen\n")


# ============================================================
# 1. READ CSV AND DISPLAY SCHEMA
# ============================================================

print("\n--- 1. Reading Data and Displaying Schema ---")

# Read sales data
sales_df = spark.read.csv(
    "sales.csv",
    header=True,
    inferSchema=True
)

# Read products data
products_df = spark.read.csv(
    "products.csv",
    header=True,
    inferSchema=True
)

# Display schema
print("\nSales Data Schema:")

sales_df.printSchema()


# ============================================================
# 2. REMOVE DUPLICATE RECORDS
# ============================================================

print("\n--- 2. Removing Duplicates ---")

print(
    "Original row count:",
    sales_df.count()
)

# Remove duplicate rows
clean_sales_df = sales_df.dropDuplicates()

print(
    "Row count after dropping duplicates:",
    clean_sales_df.count()
)

print("\nData after removing duplicates:")

clean_sales_df.show()


# ============================================================
# 3. FILTERING OPERATIONS
# ============================================================

print("\n--- 3. Filtering Operations ---")

# Remove records where quantity is 0 or less
filtered_sales_df = clean_sales_df.filter(
    col("quantity") > 0
)

print("\nSales with quantity greater than 0:")

filtered_sales_df.show()


# ============================================================
# 4. JOIN TWO DATASETS
# ============================================================

print("\n--- 4. Joining DataFrames ---")

# Join sales and products using product_id
joined_df = filtered_sales_df.join(
    products_df,
    on="product_id",
    how="inner"
)

print("\nJoined Data:")

joined_df.show()


# ============================================================
# 5. AVERAGE SALES BY PRODUCT CATEGORY
# ============================================================

print("\n--- 5. Average Sales by Product Category ---")

# Group by category
# Calculate average sale amount
# Round result to 2 decimal places

avg_sales_df = joined_df.groupBy(
    "category"
).agg(
    round(
        avg("sale_amount"),
        2
    ).alias("average_sales")
)

print("\nAverage Sales by Category:")

avg_sales_df.show()


# ============================================================
# CLEANUP
# ============================================================

spark.stop()

os.remove("sales.csv")
os.remove("products.csv")


print("=" * 60)
print("          ALL PRACTICAL 8 TASKS COMPLETED")
print("=" * 60)

             PRACTICAL 8 - PYSPARK

--- 1. Reading Data and Displaying Schema ---

Sales Data Schema:
root
 |-- transaction_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- sale_amount: double (nullable = true)


--- 2. Removing Duplicates ---
Original row count: 6
Row count after dropping duplicates: 5

Data after removing duplicates:
+--------------+----------+--------+-----------+
|transaction_id|product_id|quantity|sale_amount|
+--------------+----------+--------+-----------+
|             1|       101|       2|       50.0|
|             4|       103|       1|       30.0|
|             2|       102|       1|      120.0|
|             3|       101|       4|      100.0|
|             5|       101|       0|        0.0|
+--------------+----------+--------+-----------+


--- 3. Filtering Operations ---

Sales with quantity greater than 0:
+--------------+----------+--------+-----------+
|transaction_id|product_id|quan